# Evaluation

A unit is one reviewed build, and its truth is the newest review of it. For each unit $u$: drop the reviews of $u$ and of every reviewed build within $\sigma \ge 0.95$, answer $u$ with the rest, and compare tri's adds $A$ and removes $R$ with the reviewer's $A^{\star}$, $R^{\star}$. Counts pool over units:

$$P_+ = \frac{\sum|A\cap A^{\star}|}{\sum|A|}, \quad P_- = \frac{\sum|R\cap R^{\star}|}{\sum|R|}, \quad \mathrm{Rec} = \frac{\sum\left(|A\cap A^{\star}|+|R\cap R^{\star}|\right)}{\sum\left(|A^{\star}|+|R^{\star}|\right)}, \quad F_1 = \frac{2P\,\mathrm{Rec}}{P+\mathrm{Rec}}.$$

Review agreement answers every reviewed build with the full log and measures the share of the review's claims reproduced: it checks that a review acts at once on its own build. Every number has a bootstrap standard error over units ($B = 1000$), and a change is compared with its predecessor by a paired bootstrap on the same units:

$$\Delta = m(\text{after}) - m(\text{before}), \qquad \mathrm{se}(\Delta) = \operatorname{sd}_{\text{boot}}\left(m^{\star}(\text{after}) - m^{\star}(\text{before})\right).$$

A change is accepted only if $\Delta F_1 > 2\,\mathrm{se}(\Delta)$.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
from tri.api import evaluation as E

## Why paired errors

Two versions answered on the same units share most of their noise, so the paired error is far smaller than either standard error. The toy below draws units whose per-unit hits are correlated between versions by $\rho$; the second version is better by `lift`.

In [ ]:
def toy(units=190, lift=0.02, rho=0.9, seed=0):
    rnd = np.random.default_rng(seed)
    base = rnd.uniform(0.2, 0.6, units)
    noise = rnd.normal(0, 0.15, units)
    other = rho * noise + np.sqrt(1 - rho ** 2) * rnd.normal(0, 0.15, units)
    def rows(p, eps):
        out = []
        for pi, e in zip(p, eps):
            n = 6
            tp = int(np.clip(round(n * (pi + e)), 0, n))
            out.append(dict(tp_add=tp // 2, fp_add=(n - tp) // 2, tp_rem=tp - tp // 2, fp_rem=(n - tp) - (n - tp) // 2,
                            n_add=3, n_rem=3))
        return out
    a, b = rows(base, noise), rows(base + lift, other)
    ma, mb = E.bootstrap(a, B=300), E.bootstrap(b, B=300)
    g = E.paired_gain(a, b, B=300)
    print(f"F1 before {ma['f1']['value']:.3f} ({ma['f1']['se']:.3f}), after {mb['f1']['value']:.3f} ({mb['f1']['se']:.3f})")
    print(f"paired gain {g['gain']:+.4f} ({g['se']:.4f}) -> {'accepted' if g['gain'] > 2 * g['se'] else 'not accepted'}")

interact(toy, units=IntSlider(190, 20, 600, 10), lift=FloatSlider(value=0.02, min=-0.05, max=0.1, step=0.005),
         rho=FloatSlider(value=0.9, min=0, max=0.99, step=0.01), seed=IntSlider(0, 0, 20));

## Measured numbers

Measured on the real reviews database (198 reviews of 190 distinct builds, 114 locators; 512 adds, 620 removes, 6 iffy) and the fitted live run. Mean, standard error in brackets.

| state | $F_1$ | $P_+$ | $P_-$ | Rec | tri ops | agreement |
|---|---|---|---|---|---|---|
| v3.11.1 | 0.324 (0.016) | 0.185 (0.012) | 0.243 (0.018) | 0.683 (0.022) | 3522 | 0.955 (0.003) |
| library detached | 0.323 (0.016) | 0.184 (0.012) | 0.242 (0.018) | 0.683 (0.022) | 3530 | 0.956 (0.003) |
| one scorer | 0.375 (0.019) | 0.348 (0.023) | 0.356 (0.032) | 0.402 (0.023) | 1246 | 0.973 (0.002) |
| claims by weight × reach | 0.375 (0.019) | 0.348 (0.023) | 0.356 (0.032) | 0.402 (0.023) | 1246 | 0.976 (0.001) |
| keep supersedes on the same build only | 0.407 (0.019) | 0.350 (0.023) | 0.393 (0.030) | 0.447 (0.023) | 1311 | 0.977 (0.001) |
| directed reach | 0.408 (0.019) | 0.350 (0.023) | 0.392 (0.030) | 0.450 (0.023) | 1321 | 0.977 (0.001) |
| fitted constants ($\alpha_D = 2$) | 0.412 (0.019) | 0.359 (0.023) | 0.402 (0.031) | 0.447 (0.023) | 1278 | 0.978 (0.001) |

Reviewers make 5.9 operations per build; v3.11.1 made 18.5, now 6.7.

| change | $\Delta F_1$ |
|---|---|
| one scorer | +0.053 (0.015) |
| keep supersedes on the same build only | +0.031 (0.008) |
| directed reach ($h_\perp = 2.5h$) | +0.0008 (0.0023) |
| fitted constants | +0.0040 (0.0020) |

The old rule layers were each switched off: mined rulebook $+0.0004$ (0.0007), induced review rules $+0.023$ (0.006), family pooling $-0.005$ (0.003), consensus tells $-0.002$ (0.003), all four $+0.008$ (0.008). None gained more than two standard errors, so all four were removed. Propagation to similar cards gained nothing at $\tau \in \{0.3, 0.5, 0.7\}$ and is off.

Coordinate descent over the scorer constants accepted one step, $\alpha_D$: 3 → 2 ($+0.0040$, 0.0020); 19 other steps were rejected. $h_\parallel$ had no effect between 1.4 and 2.3: the width across the item, $h_\perp$, dominates. There are no per-item parameters.

Adding one review to the 198-review log takes 1.0 s on the live run (4.28 s on v3.11.1): 0.61 s supersession over 54,323 pair similarities and 0.55 s locator reads.

## Running it yourself (needs a run)

From the repository root, on a snapshot of the live tables:

```
python -m bot.evaluate snapshot snap.json
python -m bot.evaluate run snap.json --run runs/live --workers 2 --save rows.json
```

or from Python, on a `Tri` with its reviews (notebook 01); it takes a few minutes on 2 CPUs.

In [ ]:
RUN_EVAL = False             # set True after loading `tri` as in notebook 01
if RUN_EVAL:
    from tri import api
    m = api.evaluate(tri, workers=2, B=1000)
    print({k: (round(v["value"], 4), round(v["se"], 4)) for k, v in m.items() if isinstance(v, dict) and "se" in v})